# TSOM ablation (free GPU)

Runtime > Change runtime type > **T4 GPU**, then run the cells in order.

Four systems are compared: `amem` (baseline), `tsom_conflict` (LLM conflict labels), `tsom_time` (validity
annotations) and `tsom_both`. Everything is saved to Google Drive and every run is resumable: if Colab
disconnects, run the setup cells again (Drive, clone, Ollama), skip the checks, and re-run the cell that was
interrupted. Finished scenarios and conversations are skipped.

**Time estimates** (estimates, not measurements; A-Mem measured about 4 s per write on a T4, and the fact layer
adds one more LLM call per turn): setup about 5 min, the action suite about 10-20 min, the scenario suite about
1.5-2 h for the four systems with 40 scenarios, and the LoCoMo suite about 3 h per conversation for the four systems.

## Mount Google Drive

All results go to `OUT` on your Drive, so a disconnect loses nothing that has finished.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/g1_results/ablation'

## Choose the branch

While testing an unmerged PR set BRANCH to its branch name (e.g. eval/ablation-notebook); set it back to main for results you commit.

In [ ]:
BRANCH = "main"

## Get the code

Clones the chosen branch and prints the commit it is on. Note this commit hash: it goes next to the numbers you report.

In [ ]:
!git clone -b $BRANCH https://github.com/FatimaSohailll/tsom-agent-memory.git repo
%cd repo
!git log -1 --oneline
!pip install -q -r requirements.txt

## Start Ollama and download the model

Same as the baseline notebook. Re-run this cell after a disconnect.

In [ ]:
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:3b

## Offline tests

These use a fake model, so they check the code, not the results (about 1-2 min, estimate).

In [ ]:
!python -m pytest -q tests

## Look at the fact layer with the real model

`show_facts.py` prints, per turn, what the model extracted and how each fact was labelled, then the facts table.
`show_context.py` prints the exact context the answering model sees for "Where does Ali live now?" with
validity annotations on.

In [ ]:
!python scripts/show_facts.py --model qwen2.5:3b
!python scripts/show_context.py --model qwen2.5:3b --set use_classifier=true use_validity=true

## Sanity run: label level on the 6 hand-written scenarios

The fact layer only (no A-Mem), with LLM labels, on the first 6 reviewed scenarios, which are the hand-written
ones. A few minutes (estimate). Check that extraction recall is not near zero before starting the long runs.

In [ ]:
!python -m harness.eval_actions --config configs/exp/tsom_both.json --limit 6 --set run_name=sanity_gap1 output_dir=$OUT/sanity

## Settings

`LIMIT`: how many reviewed scenarios to use (None = all). `LOCOMO_N`: how many LoCoMo conversations for the
no-harm check (each one has about 400 turns and 200 questions).

In [ ]:
LIMIT = None
LOCOMO_N = 1
LIMIT_ARG = '' if LIMIT is None else f'--limit {LIMIT}'

## Action suite (Gap 1, label level)

Runs the fact layer twice on every scenario: with LLM labels (`gap1_llm`, shared by `tsom_conflict` and
`tsom_both`) and with newest value wins (`gap1_lww`, used by `tsom_time`). About one LLM call per turn, so
roughly 10-20 min for 40 scenarios (estimate).

In [ ]:
!python scripts/run_experiments.py --suite actions --out $OUT $LIMIT_ARG

## Scenario suite (Gap 3, answer level)

One cell per system. Each scenario turn is one A-Mem-style write, for each of the four systems, then two
questions per scenario. Estimates for 40 scenarios: about 20 min for `amem` (about 4 s per write) and about
25-30 min for each `tsom_*` system (one more LLM call per write).

In [ ]:
!python scripts/run_experiments.py --suite scenarios --configs amem --out $OUT $LIMIT_ARG

In [ ]:
!python scripts/run_experiments.py --suite scenarios --configs tsom_conflict --out $OUT $LIMIT_ARG

In [ ]:
!python scripts/run_experiments.py --suite scenarios --configs tsom_time --out $OUT $LIMIT_ARG

In [ ]:
!python scripts/run_experiments.py --suite scenarios --configs tsom_both --out $OUT $LIMIT_ARG

## LoCoMo suite (no-harm check)

One cell per system, on the first `LOCOMO_N` conversations. Estimates per conversation: about 40 min for `amem`
(about 400 writes at about 4 s, plus about 200 questions) and about 50 min for each `tsom_*` system.

In [ ]:
!python scripts/run_experiments.py --suite locomo --configs amem --conversations $LOCOMO_N --out $OUT

In [ ]:
!python scripts/run_experiments.py --suite locomo --configs tsom_conflict --conversations $LOCOMO_N --out $OUT

In [ ]:
!python scripts/run_experiments.py --suite locomo --configs tsom_time --conversations $LOCOMO_N --out $OUT

In [ ]:
!python scripts/run_experiments.py --suite locomo --configs tsom_both --conversations $LOCOMO_N --out $OUT

## Tables and verdicts

Rebuilds `tables.md` from everything on Drive (every suite cell also does this) and prints it.

In [ ]:
!python -m harness.make_tables --out $OUT > /dev/null
!cat $OUT/tables.md

## Error analysis

Wrong answers of `tsom_both`, with the context the model saw. For wrong labels, use
`--run $OUT/actions/gap1_llm` instead.

In [ ]:
!python scripts/error_dump.py --run $OUT/scenarios/tsom_both --n 15

## Save the results in the repo

Copy `tables.md` and each run's summary files from `MyDrive/g1_results/ablation/` into `results/ablation/` in the
repo, keeping the folder layout (`actions/gap1_llm/actions_summary.md` and `.json`,
`scenarios/<system>/scenarios_summary.md` and `.json`, `locomo/<system>/summary.md` and `.json`), and commit them.
Only these summary files are committed; the per-scenario and per-turn files are gitignored. Write the commit
hash printed by `git log -1 --oneline` above next to the numbers wherever you report them.